In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('Final_Train.csv')

In [3]:
df.sample(5)

,Unnamed: 0,url,Phish?
86150,86150,http://gradinaroriginal.wordpress.com/category...,0
63538,63538,http://cxyuimerdas.weebly.com,1
53111,53111,http://egywaw.ba7r.biz/t1543-topic,0
157021,157021,http://ivatlas.ru/p299291470-borba-shershnyami...,0
45084,45084,http://mangmaenter-serviso-de.blogspot.ca,1


In [4]:
df = df.drop(['Unnamed: 0'], axis = 1)

In [5]:
df

,url,Phish?
0,http://fy8.b.yahoo.com,1
1,http://neko-state.foroactivo.com/u4attachments,0
2,http://howpublishingreallyworks.blogspot.com/2...,0
3,http://marutitraders99.com/plalaa/jpn/webmai1/...,1
4,http://forum.krstarica.com/showthread.php/7556...,0
...,...,...
221396,http://mirinconceleste.blogspot.com/2016/02/,0
221397,http://www.kabelmeister.de/index.php?cat=c9131...,0
221398,http://ipfs.best-practice.se/ipfs/bafybeiet6j4...,1
221399,http://ankjyotish369.blogspot.com/2015/12/blog...,0


In [6]:
import re
import tldextract
import math
from urllib.parse import urlparse

def shannon_entropy(string):
    prob = [float(string.count(c)) / len(string) for c in dict.fromkeys(list(string))]
    entropy = -sum([p * math.log2(p) for p in prob])
    return entropy

def has_ip(domain):
    pattern = r'(\d{1,3}\.){3}\d{1,3}'
    return 1 if re.search(pattern, domain) else 0

def count_digits(s):
    return sum(c.isdigit() for c in s)

def count_letters(s):
    return sum(c.isalpha() for c in s)

def count_special_chars(s):
    return len(re.findall(r'[^a-zA-Z0-9]', s))

def count_words(s):
    words = re.split(r'[\W_]+', s)
    words = [w for w in words if w]
    return len(words)

suspicious_keywords = [
    'login', 'secure', 'update', 'bank', 'account',
    'verify', 'paypal', 'signin', 'confirm', 'free',
    'webscr', 'ebay', 'amazon'
]

def keyword_count(url):
    return sum(word in url.lower() for word in suspicious_keywords)

def extract_features(url):

    parsed = urlparse(url)
    ext = tldextract.extract(url)
    
    domain = ext.domain
    subdomain = ext.subdomain
    path = parsed.path
    
    features = {}
    
    # Basic Length Features
    features['url_length'] = len(url)
    features['domain_length'] = len(domain)
    features['subdomain_length'] = len(subdomain)
    features['path_length'] = len(path)
    
    # Count Features
    features['._count'] = url.count('.')
    features['-_count'] = url.count('-')
    features['__count'] = url.count('_')
    features['/_count'] = url.count('/')
    features['?_count'] = url.count('?')
    features['=_count'] = url.count('=')
    features['@_count'] = url.count('@')
    
    # Digit / Letter Features
    features['digit_count'] = count_digits(url)
    features['letter_count'] = count_letters(url)
    features['special_char_count'] = count_special_chars(url)
    
    # Ratio Features
    features['digit_ratio'] = features['digit_count'] / len(url)
    features['letter_ratio'] = features['letter_count'] / len(url)
    
    # Domain-based
    features['has_ip'] = has_ip(url)
    features['entropy'] = shannon_entropy(url)
    
    # Word-based
    features['word_count'] = count_words(url)
    features['keyword_count'] = keyword_count(url)
    
    # TLD suspicious
    suspicious_tlds = ['tk', 'ml', 'ga', 'cf', 'gq']
    features['suspicious_tld'] = 1 if ext.suffix in suspicious_tlds else 0
    
    # HTTPS
    features['https'] = 1 if parsed.scheme == 'https' else 0
    
    return features

def build_feature_dataframe(df, url_column):
    feature_list = df[url_column].apply(lambda x: extract_features(x))
    feature_df = pd.DataFrame(feature_list.tolist())
    return feature_df

In [7]:
data = build_feature_dataframe(df, 'url')

In [8]:
data

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,letter_count,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,https
0,22,5,5,0,3,0,0,2,0,0,...,15,6,0.045455,0.681818,0,3.663533,5,0,0,0
1,46,10,10,14,2,1,0,3,0,0,...,38,7,0.021739,0.826087,0,4.048034,6,0,0,0
2,93,8,24,49,3,4,0,5,0,0,...,74,13,0.064516,0.795699,0,4.492537,12,0,0,0
3,55,15,0,29,2,0,0,6,0,0,...,43,9,0.054545,0.781818,0,4.296837,8,0,0,0
4,109,9,5,37,3,2,0,4,1,2,...,63,14,0.293578,0.577982,0,4.943707,13,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221396,44,8,15,9,2,0,0,5,0,0,...,30,8,0.136364,0.681818,0,4.206718,6,0,0,0
221397,96,12,3,10,4,0,1,3,1,2,...,74,13,0.093750,0.770833,0,4.833953,12,0,0,0
221398,93,13,4,65,2,1,0,4,0,0,...,77,8,0.086022,0.827957,0,4.777453,7,0,0,0
221399,59,8,13,26,3,1,1,5,0,0,...,37,11,0.186441,0.627119,0,4.519870,10,0,0,0


In [9]:
data = data.drop(['https'], axis = 1)

In [10]:
data

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,digit_count,letter_count,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld
0,22,5,5,0,3,0,0,2,0,0,...,1,15,6,0.045455,0.681818,0,3.663533,5,0,0
1,46,10,10,14,2,1,0,3,0,0,...,1,38,7,0.021739,0.826087,0,4.048034,6,0,0
2,93,8,24,49,3,4,0,5,0,0,...,6,74,13,0.064516,0.795699,0,4.492537,12,0,0
3,55,15,0,29,2,0,0,6,0,0,...,3,43,9,0.054545,0.781818,0,4.296837,8,0,0
4,109,9,5,37,3,2,0,4,1,2,...,32,63,14,0.293578,0.577982,0,4.943707,13,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221396,44,8,15,9,2,0,0,5,0,0,...,6,30,8,0.136364,0.681818,0,4.206718,6,0,0
221397,96,12,3,10,4,0,1,3,1,2,...,9,74,13,0.093750,0.770833,0,4.833953,12,0,0
221398,93,13,4,65,2,1,0,4,0,0,...,8,77,8,0.086022,0.827957,0,4.777453,7,0,0
221399,59,8,13,26,3,1,1,5,0,0,...,11,37,11,0.186441,0.627119,0,4.519870,10,0,0


In [11]:
data['Phish?'] = df['Phish?']

In [12]:
data

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,letter_count,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,Phish?
0,22,5,5,0,3,0,0,2,0,0,...,15,6,0.045455,0.681818,0,3.663533,5,0,0,1
1,46,10,10,14,2,1,0,3,0,0,...,38,7,0.021739,0.826087,0,4.048034,6,0,0,0
2,93,8,24,49,3,4,0,5,0,0,...,74,13,0.064516,0.795699,0,4.492537,12,0,0,0
3,55,15,0,29,2,0,0,6,0,0,...,43,9,0.054545,0.781818,0,4.296837,8,0,0,1
4,109,9,5,37,3,2,0,4,1,2,...,63,14,0.293578,0.577982,0,4.943707,13,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221396,44,8,15,9,2,0,0,5,0,0,...,30,8,0.136364,0.681818,0,4.206718,6,0,0,0
221397,96,12,3,10,4,0,1,3,1,2,...,74,13,0.093750,0.770833,0,4.833953,12,0,0,0
221398,93,13,4,65,2,1,0,4,0,0,...,77,8,0.086022,0.827957,0,4.777453,7,0,0,1
221399,59,8,13,26,3,1,1,5,0,0,...,37,11,0.186441,0.627119,0,4.519870,10,0,0,0


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

RANDOM FOREST

In [14]:
x_train, x_test, y_train, y_test = train_test_split(
    data.iloc[:, :-1], data.iloc[:, -1], test_size=0.2, random_state=42
)

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=50,
    random_state=42,
    n_jobs=-1
)

model.fit(x_train, y_train)
y_pred = model.predict(x_test)

In [15]:
print(confusion_matrix(y_test, y_pred))
print(accuracy_score(y_test, y_pred))
print(recall_score(y_test, y_pred))

[[21420   611]
 [ 1532 20718]]
0.9516045256430523
0.9311460674157304


In [16]:
test = pd.read_csv('Final_Test.csv')

In [17]:
test

,Unnamed: 0,url,Phish?
0,0,http://computer-error-641.tk,1
1,1,http://malaysiaberih.blogspot.com/2013/02/popu...,0
2,2,http://indototo.club/humping/busty-milf-you-tu...,0
3,3,http://yummyspreadpussy.tumblr.com,1
4,4,http://sexybustybabes.blogspot.ca,1
...,...,...,...
2624521,2624521,http://megabigred.tumblr.com,1
2624522,2624522,http://hot-babe-videos.blogspot.gr,1
2624523,2624523,http://footballfanatics.com/nfl_oakland_raider...,1
2624524,2624524,http://letotrade.by/index.php?route=informatio...,0


In [18]:
test = test.drop(['Unnamed: 0'], axis = 1)

In [19]:
test_ = build_feature_dataframe(test, 'url')

In [20]:
test_['Phish?'] = test['Phish?'] 

In [21]:
test_ = test_.drop(['https'], axis = 1)

In [22]:
test_

,url_length,domain_length,subdomain_length,path_length,._count,-_count,__count,/_count,?_count,=_count,...,letter_count,special_char_count,digit_ratio,letter_ratio,has_ip,entropy,word_count,keyword_count,suspicious_tld,Phish?
0,28,18,0,0,1,2,0,2,0,0,...,19,6,0.107143,0.678571,0,3.878783,5,0,1,1
1,74,8,13,41,3,3,0,5,0,0,...,56,12,0.081081,0.756757,0,4.448941,11,0,0,0
2,52,8,0,32,2,3,0,4,0,0,...,42,10,0.000000,0.807692,0,4.159306,9,0,0,0
3,34,6,16,0,2,0,0,2,0,0,...,29,5,0.000000,0.852941,0,3.976450,4,0,0,1
4,33,8,14,0,2,0,0,2,0,0,...,28,5,0.000000,0.848485,0,3.892879,4,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2624521,28,6,10,0,2,0,0,2,0,0,...,23,5,0.000000,0.821429,0,4.039149,4,0,0,1
2624522,34,8,15,0,2,2,0,2,0,0,...,27,7,0.000000,0.794118,0,4.006437,6,0,0,1
2624523,55,16,0,28,1,0,3,3,0,0,...,47,8,0.000000,0.854545,0,4.289740,7,0,0,1
2624524,68,9,0,10,2,0,1,5,1,2,...,53,13,0.029412,0.779412,0,4.475361,12,0,0,0


In [23]:
y_pred_test = model.predict(test_.iloc[:, :-1])

In [24]:
confusion_matrix(y_pred_test, test_.iloc[:, -1])

array([[1065571,  131785],
       [  22146, 1405024]])

In [25]:
accuracy_score(y_pred_test, test_.iloc[:,-1])

0.9413490283578825

In [26]:
recall_score(y_pred_test, test_.iloc[:,-1])

0.9844825774084377

DUMPING MODEL..

In [27]:
import joblib

In [28]:
joblib.dump(model, 'model.pkl')

['model.pkl']

In [42]:
! git status

On branch Abhay
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Data_Fetching.ipynb
	modified:   Final_Train.ipynb
	modified:   Random_Forest.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.ipynb_checkpoints/
	.ipynb_checkpoints/Final_Test-checkpoint.csv
	.ipynb_checkpoints/Final_Test-checkpoint.ipynb
	.ipynb_checkpoints/Final_model-checkpoint.ipynb
	Final_Test.csv
	Final_Test.ipynb
	Final_model.ipynb
	PhishTank.csv
	Test_process.csv

no changes added to commit (use "git add" and/or "git commit -a")


In [45]:
! git add .

In [46]:
!git status

On branch Abhay
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	new file:   .ipynb_checkpoints/Final_Test-checkpoint.csv
	new file:   .ipynb_checkpoints/Final_Test-checkpoint.ipynb
	new file:   .ipynb_checkpoints/Final_model-checkpoint.ipynb
	new file:   Final_Test.csv
	new file:   Final_Test.ipynb
	modified:   Final_Train.ipynb
	new file:   Final_model.ipynb
	new file:   PhishTank.csv
	modified:   Random_Forest.ipynb
	new file:   Test_process.csv

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Data_Fetching.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.ipynb_checkpoints/



In [47]:
! git commit -m'Final_Model_Dumped'

[Abhay 5054829] 'Final_Model_Dumped'
 10 files changed, 7876652 insertions(+), 9 deletions(-)
 create mode 100644 ML/.ipynb_checkpoints/Final_Test-checkpoint.csv
 create mode 100644 ML/.ipynb_checkpoints/Final_Test-checkpoint.ipynb
 create mode 100644 ML/.ipynb_checkpoints/Final_model-checkpoint.ipynb
 create mode 100644 ML/Final_Test.csv
 create mode 100644 ML/Final_Test.ipynb
 create mode 100644 ML/Final_model.ipynb
 create mode 100644 ML/PhishTank.csv
 create mode 100644 ML/Test_process.csv


In [53]:
! git push origin Abhay

^C


error: RPC failed; HTTP 408 curl 22 The requested URL returned error: 408
send-pack: unexpected disconnect while reading sideband packet
fatal: the remote end hung up unexpectedly
Everything up-to-date


In [49]:
! git status

On branch Abhay
Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Data_Fetching.ipynb
	modified:   Final_model.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.ipynb_checkpoints/

no changes added to commit (use "git add" and/or "git commit -a")


In [59]:
! git add .

In [60]:
! git status

On branch Abhay
Changes to be committed:
  (use "git restore --staged <file>..." to unstage)
	modified:   Data_Preprocessing.ipynb
	modified:   Final_model.ipynb
	deleted:    model.pkl

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../Data_Fetching.ipynb

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	../.ipynb_checkpoints/



In [62]:
!git commit -m'Final_model.ipynb_file_added'

[Abhay c3015f4] 'Final_model.ipynb_file_added'
 3 files changed, 188 insertions(+), 39 deletions(-)
 delete mode 100644 ML/model.pkl


In [64]:
! git branch

* Abhay
  main


In [68]:
!git stash
!git checkout main

Saved working directory and index state WIP on Abhay: c3015f4 'Final_model.ipynb_file_added'


Your branch is up to date with 'origin/main'.


Switched to branch 'main'


In [69]:
! git branch

  Abhay
* main


In [70]:
! git pull origin main

Updating 7f45df9..ba3f1f7
Fast-forward
 .gitignore                      |    15 +
 Data_Fetching.ipynb             | 15684 --------------------------------------
 Dataset => Dataset/Dataset      |     0
 ML/file1.txt                    |     1 -
 MachineLearning/Hugging-face.py |    20 +
 Project_AI.pdf                  |   Bin 130753 -> 0 bytes
 abs.txt                         |     1 -
 7 files changed, 35 insertions(+), 15686 deletions(-)
 create mode 100644 .gitignore
 delete mode 100644 Data_Fetching.ipynb
 rename Dataset => Dataset/Dataset (100%)
 delete mode 100644 ML/file1.txt
 create mode 100644 MachineLearning/Hugging-face.py
 delete mode 100644 Project_AI.pdf
 delete mode 100644 abs.txt


From https://github.com/MokshJn/Phish-Secure
 * branch            main       -> FETCH_HEAD
   7f45df9..ba3f1f7  main       -> origin/main


In [73]:
! git checkout Abhay -- Final_model.ipynb Final_Train.ipynb Final_Test.ipynb Data_Preprocessing.ipynb Data_Visualisation.ipynb

In [74]:
! git branch

  Abhay
* main


In [75]:
!git add .
!git commit -m "Added important ML files from Abhay branch"

[main 0a88503] Added important ML files from Abhay branch
 5 files changed, 6205 insertions(+)
 create mode 100644 ML/Data_Preprocessing.ipynb
 create mode 100644 ML/Data_Visualisation.ipynb
 create mode 100644 ML/Final_Test.ipynb
 create mode 100644 ML/Final_Train.ipynb
 create mode 100644 ML/Final_model.ipynb


In [76]:
!git push origin main

To https://github.com/MokshJn/Phish-Secure
   ba3f1f7..0a88503  main -> main


In [88]:
! git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean


In [95]:
ls

 Volume in drive C is Windows
 Volume Serial Number is E4ED-7A6C

 Directory of C:\Users\abhay\Downloads\Phish-Secure\ML

21-03-2026  20:50    <DIR>          .
21-03-2026  20:45    <DIR>          ..
21-03-2026  20:44    <DIR>          .ipynb_checkpoints
21-03-2026  20:47            55,026 Data_Preprocessing.ipynb
21-03-2026  20:47           267,965 Data_Visualisation.ipynb
21-03-2026  20:50            87,258 Final_model.ipynb
21-03-2026  20:47            16,101 Final_Test.ipynb
21-03-2026  20:47            30,677 Final_Train.ipynb
               5 File(s)        457,027 bytes
               3 Dir(s)  206,565,421,056 bytes free


In [98]:
cd ..

C:\Users\abhay\Downloads\Phish-Secure


In [101]:
!git mv ML/mo.py MachineLearning/
!git mv ML/dataset_cleaned.csv MachineLearning/
!git mv ML/phishing_model.pkl MachineLearning/

fatal: bad source, source=ML/model.py, destination=MachineLearning/model.py
fatal: bad source, source=ML/dataset_cleaned.csv, destination=MachineLearning/dataset_cleaned.csv
fatal: bad source, source=ML/phishing_model.pkl, destination=MachineLearning/phishing_model.pkl
